# Day 1 [Retail + E-commerce + Manufacturing] Environment Setup + Data Collection

Welcome to the **Day 1 Setup** for training **Qwen-RetailEcomManufacturing** and **Llama-RetailEcomManufacturing** models.

### Objectives for Today:
1. **Environment Setup**: Mount Google Drive, install all required GPU-accelerated libraries.
2. **GPU Diagnostics**: Verify CUDA compatibility and check available VRAM.
3. **Repository Structure**: Initialize project folder structure locally and on Google Drive.
4. **Data Collection**: Download domain-specific datasets (Retail/E-Commerce & Manufacturing) from Hugging Face Hub.
5. **GDrive Backup**: Auto-sync all raw data and folder structures back to Google Drive.

---  
## Step 1: Google Drive Integration & Dependencies Installation

In [ ]:
# Mount Google Drive to persist raw datasets and fine-tuning checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install core deep learning, model quantization, and training libraries
# - transformers/datasets: Hugging Face model loading and data loading
# - accelerate/bitsandbytes: Required for 4-bit and 8-bit quantization
# - peft: Parameter-Efficient Fine-Tuning (LoRA/QLoRA)
# - wandb: Training loss & metric tracking
# - chromadb/sentence-transformers: Setup for RAG vector database (Days 9-10)

!pip install -q transformers datasets accelerate peft bitsandbytes wandb chromadb sentence-transformers

---  
## Step 2: GPU Diagnostics
Ensure that you have a GPU runtime enabled (T4, V100, or A100 is recommended).

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"[+] CUDA Available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[+] GPU Model: {gpu_name}")
    print(f"[+] Total VRAM: {total_memory:.2f} GB")
else:
    print("[-] WARNING: CUDA GPU is not detected. Please verify your Colab runtime settings:")
    print("    Go to Runtime -> Change runtime type -> Hardware accelerator -> Select 'T4 GPU' (or higher).")

---  
## Step 3: Create Project Folders
We create a local project structure in `/content/Retail/` for fast read/writes, which we will sync to Google Drive later.

In [ ]:
import os

project_dir = "/content/Retail"
folders = [
    "data/raw",
    "data/processed",
    "models",
    "src"
]

for f in folders:
    os.makedirs(os.path.join(project_dir, f), exist_ok=True)

print("[+] Project folders initialized:")
for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")

---  
## Step 4: Write & Run the Data Collection Script

In [ ]:
# Write the data collection helper module to the local src/ folder
data_collection_code = '''import os
import json
from datasets import load_dataset

def collect_datasets(raw_data_dir="data/raw"):
    os.makedirs(raw_data_dir, exist_ok=True)
    
    # 1. Retail & E-commerce Customer Support Dataset
    retail_dataset_name = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
    retail_output_path = os.path.join(raw_data_dir, "retail_ecommerce_raw.json")
    
    print(f"\\n[*] Downloading Retail & E-commerce dataset...")
    try:
        retail_ds = load_dataset(retail_dataset_name, split="train")
        retail_data = [
            {
                "instruction": row["instruction"],
                "response": row["response"],
                "category": row["category"],
                "intent": row["intent"]
            }
            for row in retail_ds
        ]
        with open(retail_output_path, "w", encoding="utf-8") as f:
            json.dump(retail_data, f, ensure_ascii=False, indent=2)
        print(f"[+] Retail dataset saved ({len(retail_data)} items)")
    except Exception as e:
        print(f"[-] Error downloading Retail dataset: {e}")
        
    # 2. Manufacturing / Lean Six Sigma Q&A Dataset
    mfg_dataset_name = "cw18/lean-six-sigma-qna-v1"
    mfg_output_path = os.path.join(raw_data_dir, "manufacturing_raw.json")
    
    print(f"\\n[*] Downloading Manufacturing & Operations dataset...")
    try:
        mfg_ds = load_dataset(mfg_dataset_name, split="train")
        mfg_data = []
        for row in mfg_ds:
            instruction = row.get("question", row.get("instruction", ""))
            response = row.get("answer", row.get("response", ""))
            if not instruction or not response:
                keys = list(row.keys())
                instruction = row[keys[0]]
                response = row[keys[1]] if len(keys) > 1 else ""
            mfg_data.append({
                "instruction": instruction,
                "response": response,
                "domain": "manufacturing_process_improvement"
            })
        with open(mfg_output_path, "w", encoding="utf-8") as f:
            json.dump(mfg_data, f, ensure_ascii=False, indent=2)
        print(f"[+] Manufacturing dataset saved ({len(mfg_data)} items)")
    except Exception as e:
        print(f"[-] Error downloading Manufacturing dataset: {e}")
'''

with open("/content/Retail/src/data_collection.py", "w") as f:
    f.write(data_collection_code)
print("[+] Created src/data_collection.py")

In [ ]:
# Execute the collection function locally
import sys
sys.path.append(project_dir)

from src.data_collection import collect_datasets
collect_datasets(raw_data_dir=os.path.join(project_dir, "data/raw"))

---  
## Step 5: Backup to Google Drive

In [ ]:
# Define your Google Drive target folder
gdrive_target_dir = "/content/drive/MyDrive/Retail"

print(f"[*] Syncing project directory to Google Drive path: {gdrive_target_dir}...")
os.makedirs(gdrive_target_dir, exist_ok=True)

# Use rsync to copy directories efficiently
!rsync -av --progress /content/Retail/ {gdrive_target_dir}/
print("[+] Google Drive backup finished successfully!")